# Aufgabe B3

In [1]:
import pandas as pd
import numpy as np
import regex as re
from collections import defaultdict as ddict
from sklearn.model_selection import train_test_split

#### a)

In [2]:
# Wörter des Body zählen nach Regex-Anweisung
def countWords(body: str,reg: str):
    result = []

    body = body.lower()
    body_words = re.split(reg, body)
    body_words_unique = list(set(body_words))

    for i in body_words_unique:
        result.append((i, body_words.count(i)))
    return result

In [3]:
# A-Priori-WK
def aPriori(x, y):
    return x/y

In [4]:
def bedingte_wahrscheinlichkeit(x,y,label):
    w_in_label = x[y==label].sum(axis=0)
    gesamt_woerter_in_label = sum(w_in_label)
    result = (w_in_label + 1) / (gesamt_woerter_in_label + len(w_in_label))
    return result

In [5]:
email_body = pd.DataFrame(pd.read_csv("./InputsB3/email_body.csv", keep_default_na=False))
email_headers = pd.DataFrame(pd.read_csv("./InputsB3/email_headers.csv"))

word_count_matrix = ddict(dict)
email_body["body"] = email_body["body"].astype('string')
email_id = 0

# filtern leerzeichen und andere Sonderzeichen heraus
for i in email_body["body"]:
    word_count_matrix["label"][email_id] = email_body.iloc[email_id]["label"]

    word_counts = countWords(i,r"\W+")
    for j in word_counts:
        word_count_matrix[j[0]][email_id] = j[1]

    email_id += 1

# Null-Werte = 0 setzen
word_count_matrix_df = pd.DataFrame(word_count_matrix).fillna(0)

In [6]:
word_count_matrix_df_dropped_labels = word_count_matrix_df.drop("label",axis=1,inplace=False)
print(word_count_matrix_df)
print(word_count_matrix_df_dropped_labels)

      label  workers  expression  been  line  com   in  today   it  vircio  \
0         0      2.0         1.0   1.0   1.0  3.0  1.0    1.0  1.0     1.0   
1         0      0.0         0.0   0.0   0.0  1.0  0.0    0.0  2.0     0.0   
2         0      0.0         0.0   0.0   0.0  1.0  3.0    0.0  0.0     0.0   
3         0      0.0         0.0   0.0   0.0  0.0  2.0    0.0  2.0     0.0   
4         0      0.0         0.0   0.0   0.0  1.0  2.0    0.0  2.0     0.0   
...     ...      ...         ...   ...   ...  ...  ...    ...  ...     ...   
2995      1      0.0         0.0   0.0   0.0  0.0  3.0    0.0  1.0     0.0   
2996      1      0.0         0.0   0.0   0.0  0.0  1.0    0.0  0.0     0.0   
2997      1      0.0         0.0   0.0   1.0  0.0  2.0    0.0  0.0     0.0   
2998      1      0.0         0.0   0.0   0.0  0.0  7.0    1.0  3.0     0.0   
2999      1      0.0         0.0   0.0   0.0  3.0  0.0    0.0  0.0     0.0   

      ...  v告e    箱  r刻知道您的   i客  w站日   而e  可以根  有number    行  

In [7]:
random_state0 = 0
X_train,X_test,y_train,y_test = train_test_split(word_count_matrix_df_dropped_labels,email_body["label"],random_state=random_state0,test_size=0.2,stratify=email_body["label"])

In [8]:
# a priori WKs für Auftreten von Spam / keinem Spam
# P(spam)
p_spam = aPriori(np.sum(y_train==1), len(y_train))
# P(no spam)
p_nospam = aPriori(np.sum(y_train==0), len(y_train))

print(f"P(Spam) = {p_spam}", f"   P(no Spam) = {p_nospam}")

P(Spam) = 0.16666666666666666    P(no Spam) = 0.8333333333333334


In [9]:
p_word_spam = bedingte_wahrscheinlichkeit(X_train, y_train, 1)
p_word_nospam = bedingte_wahrscheinlichkeit(X_train, y_train, 0)

In [10]:
print(f"P(Spam) = {p_word_spam}", f"   P(no Spam) = {p_word_nospam}")

P(Spam) = workers       0.000018
expression    0.000006
been          0.000843
line          0.000458
com           0.000771
                ...   
而e            0.000012
可以根           0.000012
有number       0.000012
行             0.000012
中步步           0.000012
Length: 36432, dtype: float64    P(no Spam) = workers       0.000211
expression    0.000048
been          0.001350
line          0.000705
com           0.001835
                ...   
而e            0.000002
可以根           0.000002
有number       0.000002
行             0.000002
中步步           0.000002
Length: 36432, dtype: float64


In [11]:
# Naive-Bayes-Klassifikator
class Bayesclassifier :
    p_spam: float
    p_nospam: float
    p_word_spam = ddict(dict)
    p_word_nospam = ddict(dict)
    x_train: any
    y_train: any
    def fit(self,x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train
        self.p_spam = aPriori(np.sum(y_train==1), len(y_train))
        self.p_nospam = aPriori(np.sum(y_train==0), len(y_train))
        self.p_word_spam = bedingte_wahrscheinlichkeit(x_train,y_train,1)
        self.p_word_nospam = bedingte_wahrscheinlichkeit(x_train,y_train,0)
        return self
    def predict(self,word_count):
        if self.x_train is None:
            print("model not Trained no prediction possible")
            return "model not Trained no prediction possible"
        log_p_spam = np.log(p_spam)
        log_p_nospam = np.log(p_nospam)
    #∏P(word|Spam) * P(Spam) / ∏P(word|no Spam) * P(no Spam)
        for j in word_count.keys():
            if j in self.x_train.columns and word_count[j] != 0.0:
                log_p_spam = log_p_spam + np.log(p_word_spam[j])
                log_p_nospam = log_p_nospam + np.log(p_word_nospam[j])
    # P(Email) = P(Email|Spam) + P(Email|no Spam)
    # p_mail = log_p_mail_spam_with_p_spam + log_p_mail_nospam_with_p_nospam
    # wir verzichten auf die normierung, da zum reinen vergleich diese nicht notwendig ist
        if log_p_spam > log_p_nospam:
            return 1
        return 0

In [12]:
bayesclassifier = Bayesclassifier().fit(X_train,y_train)
test_mail_is_spam = bayesclassifier.predict(X_test.iloc[50])
print(test_mail_is_spam)

0


In [18]:
predict_body = []
for i in range(len(X_test)):
    predict_body.append(bayesclassifier.predict(X_test.iloc[i]))
predict_body_arr = np.array(predict_body)

In [19]:
print(predict_body_arr)

[1 0 1 0 1 0 0 1 1 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 1 1 0 0 0 0 0 1 1 0 0 0 0 0 0
 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0 0 0 1
 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0 0 0
 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0
 0 1 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0
 0 0 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 1 0 0 0 1 0 0 